In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

PROJECT_PATH = '/content/drive/MyDrive/ML projects/IMDb Rating Prediction using RNN/'
%cd $PROJECT_PATH


/content/drive/MyDrive/ML projects/IMDb Rating Prediction using RNN


In [3]:
# protobuf को पूरी तरह से क्लीन करके सही वर्जन इंस्टॉल करें
!pip uninstall protobuf -y
# सभी एरर्स को ठीक करने के लिए सही और कम्पैटिबल वर्जन इंस्टॉल करें
!pip install protobuf==5.28.3 -q



Found existing installation: protobuf 5.28.3
Uninstalling protobuf-5.28.3:
  Successfully uninstalled protobuf-5.28.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.35.0 requires protobuf<5,>=3.20, but you have protobuf 5.28.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 5.28.3 which is incompatible.


In [4]:
#!pip install streamlit -q



In [5]:
!curl https://icanhazip.com


34.125.100.215


In [6]:
# %%writefile app.py

# import numpy as np
# import pandas as pd

# import streamlit as st

# from tensorflow.keras.datasets import imdb
# from tensorflow.keras.preprocessing import sequence
# from tensorflow.keras.models import load_model

# # Load the Model
# model = load_model('imdb_simple_rnn.h5')


# # Function to preprocess the user input

# word_index = imdb.get_word_index()

# def preprocess_text(text):
#   words = text.lower().split()

#   max_features=10000
#   encoded_review = []                                                          # convert the input words in indecies

#   for word in words:
#       # 1. Word index dhoondein, nahi mile toh index 2 (OOV) do
#       base_index = word_index.get(word, 2)

#       # 2. Shift index by 3 (IMDb dataset standard rule)
#       shifted_index = base_index + 3

#       # 3. FIX: Check bounds before appending to prevent GatherV2 crash
#       if shifted_index < max_features:
#           encoded_review.append(shifted_index)
#       else:
#           encoded_review.append(2) # Out of bounds numbers converted safely to OOV (2)

#   padded_review = sequence.pad_sequences([encoded_review] , maxlen=500)       # convert this indecies list in fix 500 length
#   return padded_review


# # UI Header
# st.title("IMDb movie review Sentiment Analysis")
# st.write('Enter a movie review to classify it as positive or negative.')

# # User Input
# user_input = st.text_area('Movie Review')

# if st.button('Classifity'):
#   preprocessed_input = preprocess_text(user_input)

#   #Make Prediction
#   prediction = model.predict(preprocessed_input)
#   sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

#   st.write(f"Sentiment : {sentiment}")
#   st.write(f"Prediction : {prediction[0][0]}")

# else:
#   st.write("Please enter a movie review")






%%writefile app.py
import numpy as np
import streamlit as st

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

# ----------------------------
# Page Configuration
# ----------------------------
st.set_page_config(
    page_title="IMDb Sentiment Analyzer",
    page_icon="🎬",
    layout="centered"
)

# ----------------------------
# Load Model
# ----------------------------
@st.cache_resource
def load_sentiment_model():
    return load_model("imdb_simple_rnn.h5")

model = load_sentiment_model()

# ----------------------------
# Load Word Index
# ----------------------------
word_index = imdb.get_word_index()

# ----------------------------
# Preprocessing Function
# ----------------------------
def preprocess_text(text):
    words = text.lower().split()

    max_features = 10000
    encoded_review = []

    for word in words:
        base_index = word_index.get(word, 2)
        shifted_index = base_index + 3

        if shifted_index < max_features:
            encoded_review.append(shifted_index)
        else:
            encoded_review.append(2)

    padded_review = sequence.pad_sequences(
        [encoded_review],
        maxlen=500
    )

    return padded_review


# ----------------------------
# Header
# ----------------------------
st.title("🎬 IMDb Movie Review Sentiment Analyzer")

st.markdown("""
Analyze movie reviews using a **Simple RNN with Embedding Layers**.

The model predicts whether a review expresses a:

- 😊 Positive sentiment
- 😠 Negative sentiment
""")

st.divider()

# ----------------------------
# Examples
# ----------------------------
with st.expander("📝 Example Reviews"):

    st.markdown("""
**Positive Example**

> This movie was absolutely fantastic with brilliant acting and an amazing storyline.

**Negative Example**

> The movie was boring, predictable and a complete waste of time.
""")

# ----------------------------
# User Input
# ----------------------------
review = st.text_area(
    "✍️ Enter Movie Review",
    height=180,
    placeholder="Example: This movie was absolutely amazing..."
)

# ----------------------------
# Prediction Button
# ----------------------------
if st.button(
    "🔍 Analyze Sentiment",
    use_container_width=True
):

    if review.strip() == "":
        st.warning("⚠️ Please enter a movie review.")
        st.stop()

    with st.spinner("Analyzing review sentiment..."):

        processed_input = preprocess_text(review)

        prediction = model.predict(
            processed_input,
            verbose=0
        )[0][0]

        sentiment = (
            "Positive"
            if prediction > 0.5
            else "Negative"
        )

        confidence = (
            prediction
            if prediction > 0.5
            else 1 - prediction
        )

    st.divider()

    st.subheader("📊 Analysis Result")

    if sentiment == "Positive":

        st.success(
            f"😊 Positive Review Detected"
        )

        st.progress(
            float(confidence)
        )

        st.metric(
            "Confidence Score",
            f"{confidence*100:.2f}%"
        )

    else:

        st.error(
            f"😠 Negative Review Detected"
        )

        st.progress(
            float(confidence)
        )

        st.metric(
            "Confidence Score",
            f"{confidence*100:.2f}%"
        )

    st.write("### Prediction Probability")

    st.write(
        f"Positive Probability: **{prediction*100:.2f}%**"
    )

    st.write(
        f"Negative Probability: **{(1-prediction)*100:.2f}%**"
    )

st.divider()

st.caption(
    "Built using TensorFlow, Keras, Simple RNN, Embedding Layer and Streamlit 🚀"
)


Overwriting app.py


In [7]:

%cd /content/drive/MyDrive/ML projects/IMDb Rating Prediction using RNN/


!streamlit run app.py & npx localtunnel --port 8501


/content/drive/MyDrive/ML projects/IMDb Rating Prediction using RNN
⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.100.215:8501

your url is: https://fluffy-chicken-beg.loca.lt
2026-07-09 19:56:58.503880: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-09 19:56:58.591915: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-09 19:57:04.713768: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
  Stopping...
^C


In [8]:
!git config --global user.name "chaudhary-dhruv"
!git config --global user.email "dhruv.chaudhary.63513@gmail.com"

In [9]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/ML projects/IMDb Rating Prediction using RNN/.git/


In [10]:
!git commit -m "IMDb movie review sentiment analysis"

On branch master

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	EDA and Model Training.ipynb
	Prediction.ipynb
	app.py
	imdb_simple_rnn.h5
	terminal.ipynb

nothing added to commit but untracked files present (use "git add" to track)
